In [1]:
print("Hello")

Hello


In [2]:
# Dynamic Pricing for Urban Parking Lots
# Colab‐ready, full implementation with Pathway streaming,
# three pricing models, reroute suggestions, and Bokeh dashboard.

# 1. Install dependencies
%pip install pathway pandas numpy bokeh --quiet

# 2. Imports
import math
import numpy as np
import pandas as pd
import pathway as pw
from types import SimpleNamespace

from bokeh.io import output_notebook, push_notebook, show
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure

output_notebook()

# 3. Constants & Parameters
BASE_PRICE = 10.0
MIN_MULT, MAX_MULT = 0.5, 2.0

# Model 1 tuning
α1 = 0.5           # linear occupancy sensitivity
CLIP_LOW1, CLIP_HIGH1 = 0.8, 1.2  # ±20% daily band

# Model 2 demand weights
α2, β2, γ2, δ2, ε2 = 1.0, 0.5, 0.3, 0.8, 1.2
λ2 = 0.5           # demand → price scale

# Model 3 competitor
δ_comp = 0.1       # competitor price adjustment

# Vehicle weights
VEH_W = {'car':1.0, 'bike':0.5, 'truck':1.5}

# 4. Load & preprocess static data
df = pd.read_csv('dataset.csv', parse_dates=['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

# Unique lot list
lots_list = df.LotID.unique().tolist()

# Precompute pairwise distances (Haversine)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ, dλ = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R*2*math.atan2(math.sqrt(a), math.sqrt(1-a))

locs = df[['LotID','Latitude','Longitude']].drop_duplicates().set_index('LotID')
dist_matrix = {
    i: {j: haversine(r1.Latitude, r1.Longitude, r2.Latitude, r2.Longitude)
        for j,r2 in locs.iterrows()}
    for i,r1 in locs.iterrows()
}

# 5. Build Pathway pipeline
# 5a. Source table (sorted by Timestamp)
tbl = pw.Table.from_pandas(df, timestamp="Timestamp", sort=True)

# 5b. Stateful map: compute features, three models, reroute logic
def pricing_fn(ev, state):
    # occupancy rate
    occ_rate = ev.Occupancy / ev.Capacity
    # raw demand
    D = (α2*occ_rate
         + β2 * ev.QueueLength
         - γ2 * ev.TrafficLevel
         + δ2 * int(ev.SpecialDay)
         + ε2 * VEH_W[ev.VehicleType])
    # update min/max
    state.minD = min(state.minD, D)
    state.maxD = max(state.maxD, D)
    # normalize to roughly [–0.5, +0.5]
    if state.maxD > state.minD:
        normD = (D - state.minD) / (state.maxD - state.minD) - 0.5
    else:
        normD = 0.0

    # Model 1: linear occupancy
    p1 = state.last_price1[ev.LotID] + α1 * occ_rate
    p1 = float(np.clip(
        p1,
        BASE_PRICE * CLIP_LOW1,
        BASE_PRICE * CLIP_HIGH1
    ))

    # Model 2: demand‐based
    p2 = BASE_PRICE * (1 + λ2 * normD)
    p2 = float(np.clip(p2, BASE_PRICE * MIN_MULT, BASE_PRICE * MAX_MULT))

    # Reroute suggestion if full or over‐capacity
    over = (ev.Occupancy + ev.QueueLength) >= ev.Capacity
    # find up to 3 nearest with free spots
    neighbors = sorted(
        dist_matrix[ev.LotID].items(),
        key=lambda x: x[1]
    )[1:6]  # take 5 nearest, will pick top 3 free
    reroute = []
    for nid, _ in neighbors:
        free_spots = ev.Capacity - ev.Occupancy - ev.QueueLength \
                     if nid == ev.LotID else \
                     state.free_spots.get(nid, df[df.LotID==nid].Capacity.iloc[0])
        if free_spots > 0 and len(reroute) < 3:
            reroute.append(nid)

    # Model 3: competitive adjustment
    neigh_prices = [state.last_price3[nid] for nid,_ in neighbors[:3]]
    avg_nei = np.mean(neigh_prices) if neigh_prices else BASE_PRICE
    p3 = p2
    if over and avg_nei < p2:
        p3 = p2 * (1 - δ_comp)
    elif avg_nei > p2:
        p3 = p2 * (1 + δ_comp)
    p3 = float(p3)

    # update state
    state.last_price1[ev.LotID] = p1
    state.last_price3[ev.LotID] = p3
    state.free_spots[ev.LotID] = ev.Capacity - ev.Occupancy - ev.QueueLength

    return {
        "Timestamp": ev.Timestamp,
        "LotID": ev.LotID,
        "price1": p1,
        "price2": p2,
        "price3": p3,
        "reroute": reroute
    }

priced = tbl.stateful_map(
    state_init=lambda: SimpleNamespace(
        minD=1e9,
        maxD=-1e9,
        last_price1={lot: BASE_PRICE for lot in lots_list},
        last_price3={lot: BASE_PRICE for lot in lots_list},
        free_spots={lot: df[df.LotID==lot].Capacity.iloc[0] for lot in lots_list}
    ),
    fn=pricing_fn
)

# 6. Bokeh setup: one small‐multiple per lot
cds, plots, handles = {}, {}, {}
tools = "pan,wheel_zoom,box_zoom,reset,save,hover"
for lot in lots_list:
    cds[lot] = ColumnDataSource(data=dict(
        ts=[], price1=[], price2=[], price3=[], reroute=[]
    ))
    p = figure(x_axis_type="datetime",
               width=250, height=200,
               title=f"Lot {lot}", tools=tools)
    p.line("ts", "price1", source=cds[lot], color="blue", legend_label="M1")
    p.line("ts", "price2", source=cds[lot], color="green", legend_label="M2")
    p.line("ts", "price3", source=cds[lot], color="red", legend_label="M3")
    hover = p.select_one(HoverTool)
    hover.tooltips = [
        ("Time", "@ts{%F %T}"),
        ("M1", "@price1{0.2f}"),
        ("M2", "@price2{0.2f}"),
        ("M3", "@price3{0.2f}"),
        ("Reroute to", "@reroute")
    ]
    hover.formatters = {"@ts": "datetime"}
    plots[lot] = p

grid = gridplot([plots[lot] for lot in lots_list], ncols=4, sizing_mode="scale_both")
show(grid, notebook_handle=True)

# 7. Sink: update Bokeh in real time
def update_plot(batch: pd.DataFrame):
    for _, row in batch.iterrows():
        lot = row.LotID
        new = dict(
            ts=[row.Timestamp],
            price1=[row.price1],
            price2=[row.price2],
            price3=[row.price3],
            reroute=[", ".join(map(str, row.reroute))]
        )
        cds[lot].stream(new, rollover=200)
    push_notebook()

# 8. Run the pipeline
sink = priced.to_pandas(callback=update_plot, buffer_size=1)
pw.run(sink)


[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


This is not the real Pathway package.
Visit https://pathway.com/developers/ to get Pathway.
Already tried that? Visit https://pathway.com/troubleshooting/ to get help.
Note: your platform is Windows-11-10.0.22621-SP0, your Python is CPython 3.12.4.


Loading BokehJS ...

ValueError: Missing column provided to 'parse_dates': 'Timestamp'

In [3]:
# Dynamic Pricing for Urban Parking Lots
# Full Colab‐ready implementation with:
#  • Parsing your dataset.csv columns
#  • Three pricing models + reroute suggestions
#  • Real-time streaming via Pathway
#  • Live Bokeh dashboard

# 1. Install dependencies
# !pip install pathway pandas numpy bokeh --quiet

# 2. Imports
import math
import numpy as np
import pandas as pd
import pathway as pw
from types import SimpleNamespace

from bokeh.io import output_notebook, push_notebook, show
from bokeh.layouts import gridplot
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure

output_notebook()

# 3. Load & preprocess dataset.csv
df = pd.read_csv('dataset.csv')

# Combine date + time into a single Timestamp
df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)

# Rename columns to match our code
df.rename(columns={
    'SystemCodeNumber':'LotID',
    'IsSpecialDay':'SpecialDay',
    'TrafficConditionNearby':'TrafficCondition'
}, inplace=True)

# Map traffic conditions to numeric levels
traffic_map = {'low': 1.0, 'medium': 2.0, 'high': 3.0}
df['TrafficLevel'] = df['TrafficCondition'].map(traffic_map)

# Vehicle‐type weight mapping
VEH_W = {'car': 1.0, 'bike': 0.5, 'truck': 1.5}

# Sort by time
df = df.sort_values('Timestamp').reset_index(drop=True)

# Unique list of lots
lots_list = df['LotID'].unique().tolist()

# 4. Precompute pairwise Haversine distances
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ, dλ = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

locs = df[['LotID','Latitude','Longitude']].drop_duplicates().set_index('LotID')
dist_matrix = {
    i: {
        j: haversine(r1.Latitude, r1.Longitude, r2.Latitude, r2.Longitude)
        for j,r2 in locs.iterrows()
    }
    for i,r1 in locs.iterrows()
}

# 5. Model & simulation parameters
BASE_PRICE = 10.0
MIN_MULT, MAX_MULT = 0.5, 2.0

# Model 1
α1 = 0.5
CLIP_LOW1, CLIP_HIGH1 = 0.8, 1.2

# Model 2 demand weights
α2, β2, γ2, δ2, ε2 = 1.0, 0.5, 0.3, 0.8, 1.2
λ2 = 0.5

# Model 3 competitor adjustment
δ_comp = 0.1

# 6. Build Pathway pipeline

# 6a. Source: streamed, timestamp-ordered table
tbl = pw.Table.from_pandas(df, timestamp="Timestamp", sort=True)

# 6b. Stateful map: compute features, prices, reroute
def pricing_fn(ev, state):
    # 1) Features
    occ_rate = ev.Occupancy / ev.Capacity
    D = (α2*occ_rate
         + β2 * ev.QueueLength
         - γ2 * ev.TrafficLevel
         + δ2 * int(ev.SpecialDay)
         + ε2 * VEH_W[ev.VehicleType])
    # update min/max for normalization
    state.minD = min(state.minD, D)
    state.maxD = max(state.maxD, D)
    if state.maxD > state.minD:
        normD = (D - state.minD) / (state.maxD - state.minD) - 0.5
    else:
        normD = 0.0

    # Model 1: linear occupancy
    p1 = state.last_price1[ev.LotID] + α1 * occ_rate
    p1 = float(np.clip(p1,
                       BASE_PRICE*CLIP_LOW1,
                       BASE_PRICE*CLIP_HIGH1))

    # Model 2: demand‐based
    p2 = BASE_PRICE * (1 + λ2 * normD)
    p2 = float(np.clip(p2,
                       BASE_PRICE*MIN_MULT,
                       BASE_PRICE*MAX_MULT))

    # Reroute if overburdened
    over = (ev.Occupancy + ev.QueueLength) >= ev.Capacity
    neighbors = sorted(dist_matrix[ev.LotID].items(),
                       key=lambda x: x[1])[1:6]
    reroute = []
    for nid,_ in neighbors:
        free_here = state.free_spots.get(nid,
                          df[df.LotID==nid].Capacity.iloc[0]
                        )  # previous free estimate
        if free_here > 0 and len(reroute) < 3:
            reroute.append(nid)

    # Model 3: competitive adjustment
    neigh_prices = [state.last_price3[nid] for nid,_ in neighbors[:3]]
    avg_nei = np.mean(neigh_prices) if neigh_prices else BASE_PRICE
    p3 = p2
    if over and avg_nei < p2:
        p3 = p2 * (1 - δ_comp)
    elif avg_nei > p2:
        p3 = p2 * (1 + δ_comp)
    p3 = float(p3)

    # Update state
    state.last_price1[ev.LotID] = p1
    state.last_price3[ev.LotID] = p3
    state.free_spots[ev.LotID] = max(
        0, ev.Capacity - ev.Occupancy - ev.QueueLength
    )

    return {
        "Timestamp": ev.Timestamp,
        "LotID": ev.LotID,
        "price1": p1,
        "price2": p2,
        "price3": p3,
        "reroute": reroute
    }

priced = tbl.stateful_map(
    state_init=lambda: SimpleNamespace(
        minD=1e9,
        maxD=-1e9,
        last_price1={lot: BASE_PRICE for lot in lots_list},
        last_price3={lot: BASE_PRICE for lot in lots_list},
        free_spots={lot: df[df.LotID==lot].Capacity.iloc[0]
                    for lot in lots_list}
    ),
    fn=pricing_fn
)

# 7. Bokeh dashboard setup
cds, plots = {}, {}
tools = "pan,wheel_zoom,box_zoom,reset,save,hover"

for lot in lots_list:
    cds[lot] = ColumnDataSource(data=dict(
        ts=[], price1=[], price2=[], price3=[], reroute=[]
    ))
    p = figure(x_axis_type="datetime",
               width=250, height=200,
               title=f"Lot {lot}", tools=tools)
    p.line("ts","price1", source=cds[lot], color="blue", legend_label="M1")
    p.line("ts","price2", source=cds[lot], color="green", legend_label="M2")
    p.line("ts","price3", source=cds[lot], color="red", legend_label="M3")
    hover = p.select_one(HoverTool)
    hover.tooltips = [
        ("Time", "@ts{%F %T}"),
        ("M1", "@price1{0.2f}"),
        ("M2", "@price2{0.2f}"),
        ("M3", "@price3{0.2f}"),
        ("Reroute", "@reroute")
    ]
    hover.formatters = {"@ts": "datetime"}
    plots[lot] = p

grid = gridplot([plots[lot] for lot in lots_list], ncols=4,
                sizing_mode="scale_both")
show(grid, notebook_handle=True)

# 8. Sink callback: update Bokeh streams
def update_plot(batch: pd.DataFrame):
    for _, row in batch.iterrows():
        lot = row.LotID
        cds[lot].stream({
            'ts': [row.Timestamp],
            'price1': [row.price1],
            'price2': [row.price2],
            'price3': [row.price3],
            'reroute': [", ".join(map(str, row.reroute))]
        }, rollover=200)
    push_notebook()

# 9. Execute streaming pipeline
sink = priced.to_pandas(callback=update_plot, buffer_size=1)
pw.run(sink)

# Now your notebook will run a live simulation, updating prices and reroute suggestions.

Loading BokehJS ...

This is not the real Pathway package.
Visit https://pathway.com/developers/ to get Pathway.
Already tried that? Visit https://pathway.com/troubleshooting/ to get help.
Note: your platform is Windows-11-10.0.22621-SP0, your Python is CPython 3.12.4.


AttributeError: module 'pathway' has no attribute 'Table'
This is not the real Pathway package.
Visit https://pathway.com/developers/ to get Pathway.
Already tried that? Visit https://pathway.com/troubleshooting/ to get help.
Note: your platform is Windows-11-10.0.22621-SP0, your Python is CPython 3.12.4.

In [4]:
# Dynamic Pricing Simulation Without Pathway
# ------------------------------------------
# Simulates real‐time dynamic pricing for 14 parking lots using Python loops
# and Bokeh for live visualization. Includes:
#  • Model 1: linear occupancy
#  • Model 2: demand‐based
#  • Model 3: competitive + reroute suggestions

# 1. Install dependencies
# !pip install pandas numpy bokeh --quiet

# 2. Imports & notebook setup
import pandas as pd
import numpy as np
import math
import time

from bokeh.io import output_notebook, show, push_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure
from bokeh.layouts import gridplot

output_notebook()

# 3. Load & preprocess dataset
df = pd.read_csv('dataset.csv')

# combine date/time → Timestamp
df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)

# rename columns to match code
df.rename(columns={
    'SystemCodeNumber': 'LotID',
    'TrafficConditionNearby': 'TrafficCondition',
    'IsSpecialDay': 'SpecialDay'
}, inplace=True)

# map traffic to numeric
traffic_map = {'low': 1.0, 'medium': 2.0, 'high': 3.0}
df['TrafficLevel'] = df['TrafficCondition'].map(traffic_map)

# sort by time
df.sort_values('Timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)

# unique lots
lots = df['LotID'].unique().tolist()

# 4. Precompute pairwise Haversine distances
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ, dλ = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

locs = df[['LotID','Latitude','Longitude']].drop_duplicates().set_index('LotID')
dist_matrix = {
    i: {
        j: haversine(r1.Latitude, r1.Longitude, r2.Latitude, r2.Longitude)
        for j,r2 in locs.iterrows()
    }
    for i,r1 in locs.iterrows()
}

# 5. Parameters & initial state
BASE_PRICE = 10.0
MIN_MULT, MAX_MULT = 0.5, 2.0

α1 = 0.5                # Model 1 occupancy sensitivity
CLIP_LOW1, CLIP_HIGH1 = 0.8, 1.2

α2, β2, γ2, δ2, ε2 = 1.0, 0.5, 0.3, 0.8, 1.2   # Model 2 weights
λ2 = 0.5                # demand→price scale

δ_comp = 0.1            # competitor adjustment

VEH_W = {'car':1.0, 'bike':0.5, 'truck':1.5}

# per‐lot state dicts
state = {
    'last_price1': {lot: BASE_PRICE for lot in lots},
    'last_price3': {lot: BASE_PRICE for lot in lots},
    'minD':        {lot: 1e9 for lot in lots},
    'maxD':        {lot: -1e9 for lot in lots},
    'free_spots':  {}
}

# initialize free_spots from first record of each lot
first = df.groupby('LotID').first().reset_index()
for _, r in first.iterrows():
    state['free_spots'][r.LotID] = max(
        0, r.Capacity - r.Occupancy - r.QueueLength
    )

# 6. Price & reroute computation
def compute_pricing(row):
    lot = row.LotID
    # occupancy rate
    occ_rate = row.Occupancy / row.Capacity

    # raw demand
    D = (α2*occ_rate
         + β2 * row.QueueLength
         - γ2 * row.TrafficLevel
         + δ2 * int(row.SpecialDay)
         + ε2 * VEH_W[row.VehicleType])

    # update min/max for this lot
    state['minD'][lot] = min(state['minD'][lot], D)
    state['maxD'][lot] = max(state['maxD'][lot], D)

    # normalized demand ∈ [–0.5, +0.5]
    if state['maxD'][lot] > state['minD'][lot]:
        normD = ((D - state['minD'][lot]) /
                 (state['maxD'][lot] - state['minD'][lot]) - 0.5)
    else:
        normD = 0.0

    # Model 1: linear occupancy
    p1 = state['last_price1'][lot] + α1 * occ_rate
    p1 = float(np.clip(p1,
                       BASE_PRICE*CLIP_LOW1,
                       BASE_PRICE*CLIP_HIGH1))

    # Model 2: demand‐based
    p2 = BASE_PRICE * (1 + λ2 * normD)
    p2 = float(np.clip(p2,
                       BASE_PRICE*MIN_MULT,
                       BASE_PRICE*MAX_MULT))

    # detect overburden
    over = (row.Occupancy + row.QueueLength) >= row.Capacity

    # find nearest 5 for reroute + competitor
    neighbors = sorted(
        dist_matrix[lot].items(), key=lambda x: x[1]
    )[1:6]

    # reroute suggestions (up to 3 with free spots)
    reroute = []
    for nid,_ in neighbors:
        if state['free_spots'].get(nid, 0) > 0 and len(reroute)<3:
            reroute.append(nid)

    # Model 3: competitive
    neigh_prices = [state['last_price3'][nid] for nid,_ in neighbors[:3]]
    avg_nei = np.mean(neigh_prices) if neigh_prices else BASE_PRICE

    if over and avg_nei < p2:
        p3 = p2 * (1 - δ_comp)
    elif avg_nei > p2:
        p3 = p2 * (1 + δ_comp)
    else:
        p3 = p2
    p3 = float(p3)

    # update state
    state['last_price1'][lot] = p1
    state['last_price3'][lot] = p3
    state['free_spots'][lot] = max(
        0, row.Capacity - row.Occupancy - row.QueueLength
    )

    return p1, p2, p3, reroute

# 7. Bokeh dashboard setup
cds = {}
plots = []
TOOLS = "pan,wheel_zoom,box_zoom,reset,save,hover"

for lot in lots:
    cds[lot] = ColumnDataSource(data=dict(
        ts=[], p1=[], p2=[], p3=[], reroute=[]
    ))
    p = figure(x_axis_type="datetime",
               width=250, height=200,
               title=f"Lot {lot}", tools=TOOLS)
    p.line('ts', 'p1', source=cds[lot], color='blue', legend_label='M1')
    p.line('ts', 'p2', source=cds[lot], color='green', legend_label='M2')
    p.line('ts', 'p3', source=cds[lot], color='red', legend_label='M3')

    hover = p.select_one(HoverTool)
    hover.tooltips = [
        ("Time", "@ts{%F %T}"),
        ("M1", "@p1{0.2f}"), ("M2", "@p2{0.2f}"),
        ("M3", "@p3{0.2f}"), ("Reroute", "@reroute")
    ]
    hover.formatters = {"@ts": "datetime"}
    plots.append(p)

grid = gridplot(plots, ncols=4, sizing_mode="scale_both")
handle = show(grid, notebook_handle=True)

# 8. Real‐time simulation loop
for _, row in df.iterrows():
    p1, p2, p3, reroute = compute_pricing(row)
    lot = row.LotID

    new = dict(
        ts=[row.Timestamp],
        p1=[p1], p2=[p2], p3=[p3],
        reroute=[", ".join(reroute)]
    )
    cds[lot].stream(new, rollover=200)
    push_notebook(handle=handle)
    time.sleep(0.05)  # simulate real‐time delay

Loading BokehJS ...

KeyError: 'cycle'

In [5]:
# Dynamic Pricing Simulation Without Pathway
# ------------------------------------------
# Models 1–3 + reroute, live‐updating Bokeh dashboard.

# 1. Install deps
# !pip install pandas numpy bokeh --quiet

# 2. Imports
import pandas as pd
import numpy as np
import math, time

from bokeh.io import output_notebook, show, push_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure
from bokeh.layouts import gridplot

output_notebook()

# 3. Load & preprocess
df = pd.read_csv('dataset.csv')

# combine date+time → Timestamp
df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)

# rename for consistency
df.rename(columns={
    'SystemCodeNumber': 'LotID',
    'TrafficConditionNearby': 'TrafficCondition',
    'IsSpecialDay': 'SpecialDay'
}, inplace=True)

# map traffic to numeric
traffic_map = {'low':1.0, 'medium':2.0, 'high':3.0}
df['TrafficLevel'] = df['TrafficCondition'].map(traffic_map)

# sort, reset
df.sort_values('Timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)

# unique lots
lots = df['LotID'].unique().tolist()

# 4. Haversine distances
def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ, dλ = math.radians(lat2-lat1), math.radians(lon2-lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

locs = df[['LotID','Latitude','Longitude']].drop_duplicates().set_index('LotID')
dist_matrix = {
    i: {j: haversine(r1.Latitude, r1.Longitude, r2.Latitude, r2.Longitude)
        for j,r2 in locs.iterrows()}
    for i,r1 in locs.iterrows()
}

# 5. Params & initial state
BASE_PRICE = 10.0
MIN_MULT, MAX_MULT = 0.5, 2.0

α1 = 0.5
CLIP_LOW1, CLIP_HIGH1 = 0.8, 1.2

α2, β2, γ2, δ2, ε2 = 1.0, 0.5, 0.3, 0.8, 1.2
λ2 = 0.5

δ_comp = 0.1

# vehicle weights, with default
VEH_W = {'car':1.0, 'bike':0.5, 'truck':1.5}
DEFAULT_VEH_W = 1.0

state = {
    'last_price1': {lot: BASE_PRICE for lot in lots},
    'last_price3': {lot: BASE_PRICE for lot in lots},
    'minD':        {lot: 1e9 for lot in lots},
    'maxD':        {lot: -1e9 for lot in lots},
    'free_spots':  {}
}

# initialize free_spots
first = df.groupby('LotID').first().reset_index()
for _, r in first.iterrows():
    state['free_spots'][r.LotID] = max(
        0, r.Capacity - r.Occupancy - r.QueueLength
    )

# 6. Pricing + reroute function
def compute_pricing(row):
    lot = row.LotID
    occ_rate = row.Occupancy / row.Capacity

    # raw demand
    veh_wt = VEH_W.get(row.VehicleType, DEFAULT_VEH_W)
    D = (α2*occ_rate
         + β2*row.QueueLength
         - γ2*row.TrafficLevel
         + δ2*int(row.SpecialDay)
         + ε2*veh_wt)

    # update min/max
    state['minD'][lot] = min(state['minD'][lot], D)
    state['maxD'][lot] = max(state['maxD'][lot], D)

    # normalize ∈[–0.5,+0.5]
    if state['maxD'][lot] > state['minD'][lot]:
        normD = (D - state['minD'][lot]) / (state['maxD'][lot] - state['minD'][lot]) - 0.5
    else:
        normD = 0.0

    # Model 1
    p1 = state['last_price1'][lot] + α1 * occ_rate
    p1 = float(np.clip(p1, BASE_PRICE*CLIP_LOW1, BASE_PRICE*CLIP_HIGH1))

    # Model 2
    p2 = BASE_PRICE * (1 + λ2 * normD)
    p2 = float(np.clip(p2, BASE_PRICE*MIN_MULT, BASE_PRICE*MAX_MULT))

    # overburden?
    over = (row.Occupancy + row.QueueLength) >= row.Capacity
    neighbors = sorted(dist_matrix[lot].items(), key=lambda x: x[1])[1:6]

    # reroute up to 3 with free spots
    reroute = []
    for nid,_ in neighbors:
        if state['free_spots'].get(nid, 0) > 0 and len(reroute)<3:
            reroute.append(nid)

    # Model 3 competitor
    neigh_prices = [state['last_price3'][nid] for nid,_ in neighbors[:3]]
    avg_nei = np.mean(neigh_prices) if neigh_prices else BASE_PRICE

    if over and avg_nei < p2:
        p3 = p2 * (1 - δ_comp)
    elif avg_nei > p2:
        p3 = p2 * (1 + δ_comp)
    else:
        p3 = p2
    p3 = float(p3)

    # update state
    state['last_price1'][lot] = p1
    state['last_price3'][lot] = p3
    state['free_spots'][lot] = max(0, row.Capacity - row.Occupancy - row.QueueLength)

    return p1, p2, p3, reroute

# 7. Bokeh setup
cds = {}
plots = []
TOOLS = "pan,wheel_zoom,box_zoom,reset,save,hover"

for lot in lots:
    cds[lot] = ColumnDataSource(data=dict(ts=[], p1=[], p2=[], p3=[], reroute=[]))
    p = figure(x_axis_type="datetime", width=250, height=200, title=f"Lot {lot}", tools=TOOLS)
    p.line('ts','p1', source=cds[lot], color='blue',  legend_label='M1')
    p.line('ts','p2', source=cds[lot], color='green',legend_label='M2')
    p.line('ts','p3', source=cds[lot], color='red',   legend_label='M3')
    hover = p.select_one(HoverTool)
    hover.tooltips = [
      ("Time","@ts{%F %T}"),("M1","@p1{0.2f}"),("M2","@p2{0.2f}"),
      ("M3","@p3{0.2f}"),("Reroute","@reroute")
    ]
    hover.formatters = {"@ts":"datetime"}
    plots.append(p)

grid = gridplot(plots, ncols=4, sizing_mode="scale_both")
handle = show(grid, notebook_handle=True)

# 8. Simulation loop
for _, row in df.iterrows():
    p1,p2,p3,reroute = compute_pricing(row)
    lot = row.LotID
    cds[lot].stream({
        'ts':      [row.Timestamp],
        'p1':      [p1],
        'p2':      [p2],
        'p3':      [p3],
        'reroute': [", ".join(reroute)]
    }, rollover=200)
    push_notebook(handle=handle)
    time.sleep(0.05)

Loading BokehJS ...

KeyboardInterrupt: 

In [6]:
# Dynamic Pricing Simulation — Batched Bokeh Updates (No Pathway)
# --------------------------------------------------------------
# Implements Models 1–3 + reroute suggestions.
# Batches plot updates every 100 rows to run quickly.

# 1. Install dependencies
# !pip install pandas numpy bokeh --quiet

# 2. Imports
import pandas as pd
import numpy as np
import math

from bokeh.io import output_notebook, show, push_notebook
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.plotting import figure
from bokeh.layouts import gridplot

output_notebook()

# 3. Load & preprocess dataset
df = pd.read_csv('dataset.csv')

# Combine date+time → Timestamp
df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)

# Rename columns
df.rename(columns={
    'SystemCodeNumber': 'LotID',
    'TrafficConditionNearby': 'TrafficCondition',
    'IsSpecialDay': 'SpecialDay'
}, inplace=True)

# Map traffic to numeric
traffic_map = {'low': 1.0, 'medium': 2.0, 'high': 3.0}
df['TrafficLevel'] = df['TrafficCondition'].map(traffic_map)

# Sort by time
df.sort_values('Timestamp', inplace=True)
df.reset_index(drop=True, inplace=True)

# Unique lot IDs
lots = df['LotID'].unique().tolist()

# 4. Precompute Haversine distances
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ = math.radians(lat2 - lat1)
    dλ = math.radians(lon2 - lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

locs = df[['LotID','Latitude','Longitude']].drop_duplicates().set_index('LotID')
dist_matrix = {
    i: {
        j: haversine(r1.Latitude, r1.Longitude, r2.Latitude, r2.Longitude)
        for j, r2 in locs.iterrows()
    }
    for i, r1 in locs.iterrows()
}

# 5. Model parameters & state
BASE_PRICE = 10.0
MIN_MULT, MAX_MULT = 0.5, 2.0

# Model 1
α1 = 0.5
CLIP_LOW1, CLIP_HIGH1 = 0.8, 1.2

# Model 2
α2, β2, γ2, δ2, ε2 = 1.0, 0.5, 0.3, 0.8, 1.2
λ2 = 0.5

# Model 3
δ_comp = 0.1

# Vehicle weights (default for unknown)
VEH_W = {'car': 1.0, 'bike': 0.5, 'truck': 1.5}
DEFAULT_VEH_W = 1.0

# State dictionaries
state = {
    'last_price1': {lot: BASE_PRICE for lot in lots},
    'last_price3': {lot: BASE_PRICE for lot in lots},
    'minD':        {lot: 1e9 for lot in lots},
    'maxD':        {lot: -1e9 for lot in lots},
    'free_spots':  {}
}

# Initialize free_spots from first record per lot
first = df.groupby('LotID').first().reset_index()
for _, r in first.iterrows():
    state['free_spots'][r.LotID] = max(
        0, r.Capacity - r.Occupancy - r.QueueLength
    )

# 6. Pricing + reroute function
def compute_pricing(row):
    lot = row.LotID
    occ_rate = row.Occupancy / row.Capacity

    # Raw demand
    veh_wt = VEH_W.get(row.VehicleType, DEFAULT_VEH_W)
    D = (α2*occ_rate
         + β2*row.QueueLength
         - γ2*row.TrafficLevel
         + δ2*int(row.SpecialDay)
         + ε2*veh_wt)

    # Update min/max for normalization
    state['minD'][lot] = min(state['minD'][lot], D)
    state['maxD'][lot] = max(state['maxD'][lot], D)
    if state['maxD'][lot] > state['minD'][lot]:
        normD = (D - state['minD'][lot]) / (state['maxD'][lot] - state['minD'][lot]) - 0.5
    else:
        normD = 0.0

    # Model 1: linear occupancy
    p1 = state['last_price1'][lot] + α1 * occ_rate
    p1 = float(np.clip(p1, BASE_PRICE*CLIP_LOW1, BASE_PRICE*CLIP_HIGH1))

    # Model 2: demand-based
    p2 = BASE_PRICE * (1 + λ2 * normD)
    p2 = float(np.clip(p2, BASE_PRICE*MIN_MULT, BASE_PRICE*MAX_MULT))

    # Check overburden
    over = (row.Occupancy + row.QueueLength) >= row.Capacity
    neighbors = sorted(dist_matrix[lot].items(), key=lambda x: x[1])[1:6]

    # Reroute suggestions
    reroute = []
    for nid, _ in neighbors:
        if state['free_spots'].get(nid, 0) > 0 and len(reroute) < 3:
            reroute.append(nid)

    # Model 3: competitive adjustment
    neigh_prices = [state['last_price3'][nid] for nid, _ in neighbors[:3]]
    avg_nei = np.mean(neigh_prices) if neigh_prices else BASE_PRICE
    if over and avg_nei < p2:
        p3 = p2 * (1 - δ_comp)
    elif avg_nei > p2:
        p3 = p2 * (1 + δ_comp)
    else:
        p3 = p2
    p3 = float(p3)

    # Update state
    state['last_price1'][lot] = p1
    state['last_price3'][lot] = p3
    state['free_spots'][lot] = max(0, row.Capacity - row.Occupancy - row.QueueLength)

    return p1, p2, p3, reroute

# 7. Bokeh dashboard setup
cds = {}
plots = []
TOOLS = "pan,wheel_zoom,box_zoom,reset,save,hover"

for lot in lots:
    cds[lot] = ColumnDataSource(data=dict(ts=[], p1=[], p2=[], p3=[], reroute=[]))
    p = figure(x_axis_type="datetime", width=250, height=200,
               title=f"Lot {lot}", tools=TOOLS)
    p.line('ts','p1', source=cds[lot], color='blue',  legend_label='M1')
    p.line('ts','p2', source=cds[lot], color='green',legend_label='M2')
    p.line('ts','p3', source=cds[lot], color='red',   legend_label='M3')
    hover = p.select_one(HoverTool)
    hover.tooltips = [
        ("Time","@ts{%F %T}"),
        ("M1","@p1{0.2f}"), ("M2","@p2{0.2f}"),
        ("M3","@p3{0.2f}"), ("Reroute","@reroute")
    ]
    hover.formatters = {"@ts":"datetime"}
    plots.append(p)

grid = gridplot(plots, ncols=4, sizing_mode="scale_both")
handle = show(grid, notebook_handle=True)

# 8. Batch‐flush function
def _flush(batch):
    updates = {lot: {'ts':[], 'p1':[], 'p2':[], 'p3':[], 'reroute':[]} for lot in lots}
    for ts, lot, p1, p2, p3, r in batch:
        updates[lot]['ts'].append(ts)
        updates[lot]['p1'].append(p1)
        updates[lot]['p2'].append(p2)
        updates[lot]['p3'].append(p3)
        updates[lot]['reroute'].append(", ".join(map(str, r)))
    for lot, data in updates.items():
        if data['ts']:
            cds[lot].stream(data, rollover=200)
    push_notebook(handle=handle)

# 9. Simulation loop with batch updates
batch = []
BATCH_SIZE = 100

for idx, row in df.iterrows():
    p1, p2, p3, reroute = compute_pricing(row)
    batch.append((row.Timestamp, row.LotID, p1, p2, p3, reroute))
    if (idx + 1) % BATCH_SIZE == 0:
        _flush(batch)
        batch.clear()

# Final flush
if batch:
    _flush(batch)

Loading BokehJS ...

In [7]:
# Dynamic Pricing Simulation (Static Precompute + Grid Plot)
# ---------------------------------------------------------
# Computes Models 1–3 and reroute suggestions for all time‐steps,
# then renders static line charts for each lot in a Bokeh grid.

# 1. Install dependencies
# !pip install pandas numpy bokeh --quiet

# 2. Imports & notebook setup
import pandas as pd
import numpy as np
import math

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.layouts import gridplot

output_notebook()

# 3. Load & preprocess dataset
df = pd.read_csv('dataset.csv')

# combine date+time → Timestamp
df['Timestamp'] = pd.to_datetime(
    df['LastUpdatedDate'] + ' ' + df['LastUpdatedTime'],
    format='%d-%m-%Y %H:%M:%S'
)

# rename columns
df.rename(columns={
    'SystemCodeNumber': 'LotID',
    'TrafficConditionNearby': 'TrafficCondition',
    'IsSpecialDay': 'SpecialDay'
}, inplace=True)

# map traffic to numeric
traffic_map = {'low':1.0, 'medium':2.0, 'high':3.0}
df['TrafficLevel'] = df['TrafficCondition'].map(traffic_map)

# sort by time
df = df.sort_values('Timestamp').reset_index(drop=True)

# list of unique lots
lots = df['LotID'].unique().tolist()

# 4. Precompute pairwise Haversine distances
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ = math.radians(lat2 - lat1)
    dλ = math.radians(lon2 - lon1)
    a = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

locs = df[['LotID','Latitude','Longitude']].drop_duplicates().set_index('LotID')
dist_matrix = {
    i: {
        j: haversine(r1.Latitude, r1.Longitude, r2.Latitude, r2.Longitude)
        for j,r2 in locs.iterrows()
    }
    for i,r1 in locs.iterrows()
}

# 5. Model parameters & state
BASE_PRICE = 10.0
MIN_MULT, MAX_MULT = 0.5, 2.0

# Model 1
α1 = 0.5
CLIP_LOW1, CLIP_HIGH1 = 0.8, 1.2

# Model 2 weights
α2, β2, γ2, δ2, ε2 = 1.0, 0.5, 0.3, 0.8, 1.2
λ2 = 0.5

# Model 3
δ_comp = 0.1

# Vehicle weights with default
VEH_W = {'car':1.0, 'bike':0.5, 'truck':1.5}
DEFAULT_VEH_W = 1.0

# initialize state dictionaries
state = {
    'last_price1': {lot: BASE_PRICE for lot in lots},
    'last_price3': {lot: BASE_PRICE for lot in lots},
    'minD':        {lot: 1e9 for lot in lots},
    'maxD':        {lot: -1e9 for lot in lots},
    'free_spots':  {}
}

# set initial free spots from first record of each lot
first = df.groupby('LotID').first().reset_index()
for _, r in first.iterrows():
    state['free_spots'][r.LotID] = max(
        0, r.Capacity - r.Occupancy - r.QueueLength
    )

# 6. Pricing + reroute function
def compute_pricing(row):
    lot = row.LotID
    occ_rate = row.Occupancy / row.Capacity

    # raw demand
    wt = VEH_W.get(row.VehicleType, DEFAULT_VEH_W)
    D = (α2*occ_rate
         + β2*row.QueueLength
         - γ2*row.TrafficLevel
         + δ2*int(row.SpecialDay)
         + ε2*wt)

    # update min/max
    state['minD'][lot] = min(state['minD'][lot], D)
    state['maxD'][lot] = max(state['maxD'][lot], D)
    if state['maxD'][lot] > state['minD'][lot]:
        normD = ((D - state['minD'][lot]) /
                 (state['maxD'][lot] - state['minD'][lot]) - 0.5)
    else:
        normD = 0.0

    # Model 1: linear
    p1 = state['last_price1'][lot] + α1 * occ_rate
    p1 = float(np.clip(p1, BASE_PRICE*CLIP_LOW1, BASE_PRICE*CLIP_HIGH1))

    # Model 2: demand‐based
    p2 = BASE_PRICE * (1 + λ2 * normD)
    p2 = float(np.clip(p2, BASE_PRICE*MIN_MULT, BASE_PRICE*MAX_MULT))

    # overburdened?
    over = (row.Occupancy + row.QueueLength) >= row.Capacity
    neighbors = sorted(dist_matrix[lot].items(), key=lambda x: x[1])[1:6]

    # reroute: up to 3 with free spots
    reroute = []
    for nid,_ in neighbors:
        if state['free_spots'].get(nid, 0) > 0 and len(reroute) < 3:
            reroute.append(nid)

    # Model 3: competitive
    neigh_prices = [state['last_price3'][nid] for nid,_ in neighbors[:3]]
    avg_nei = np.mean(neigh_prices) if neigh_prices else BASE_PRICE

    if over and avg_nei < p2:
        p3 = p2 * (1 - δ_comp)
    elif avg_nei > p2:
        p3 = p2 * (1 + δ_comp)
    else:
        p3 = p2
    p3 = float(p3)

    # update state
    state['last_price1'][lot] = p1
    state['last_price3'][lot] = p3
    state['free_spots'][lot] = max(
        0, row.Capacity - row.Occupancy - row.QueueLength
    )

    return p1, p2, p3, reroute

# 7. Compute prices for all rows
p1_list, p2_list, p3_list, reroute_list = [], [], [], []
for _, row in df.iterrows():
    p1, p2, p3, rer = compute_pricing(row)
    p1_list.append(p1)
    p2_list.append(p2)
    p3_list.append(p3)
    reroute_list.append(", ".join(rer))

df['price1']  = p1_list
df['price2']  = p2_list
df['price3']  = p3_list
df['reroute'] = reroute_list

# 8. Static grid plot of all lots
plots = []
TOOLS = "pan,wheel_zoom,box_zoom,reset,save,hover"

for lot in lots:
    sub = df[df.LotID == lot]
    p = figure(x_axis_type="datetime",
               width=300, height=250,
               title=f"Lot {lot}", tools=TOOLS)
    p.line(sub.Timestamp, sub.price1, color="blue",  legend_label="M1")
    p.line(sub.Timestamp, sub.price2, color="green",legend_label="M2")
    p.line(sub.Timestamp, sub.price3, color="red",   legend_label="M3")
    hover = p.select_one(HoverTool)
    hover.tooltips = [
        ("Time",   "@x{%F %T}"),
        ("M1",     "@y{0.2f}")  # shows whichever line is hovered
    ]
    hover.formatters = {"@x": "datetime"}
    p.legend.location = "top_left"
    plots.append(p)

grid = gridplot(plots, ncols=4, sizing_mode="scale_both")
show(grid)

Loading BokehJS ...